<a href="https://colab.research.google.com/github/useanynoms-commits/IDRA/blob/main/Assignment_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Processed E-commerce Dataset
Combining Orders, Customers, and Products data
"""

import pandas as pd

# Load datasets
orders = pd.read_csv('Day9_Orders.csv')
customers = pd.read_csv('Day9_Customers.csv')
products = pd.read_csv('Day9_Products.csv')

print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)

Orders: (120, 7)
Customers: (30, 5)
Products: (20, 5)


In [2]:
# Merging datasets
merged = pd.merge(orders, products, on='Product_ID')
merged = pd.merge(merged, customers, on='Customer_ID')
print("Merged shape:", merged.shape)
print(merged.head())

Merged shape: (120, 15)
  Order_ID  Order_Date Customer_ID Product_ID  Quantity    Payment_Method  \
0    O0001  2026-02-19        C027       P019         2       Credit Card   
1    O0002  2026-01-25        C006       P003         2        Debit Card   
2    O0003  2026-02-26        C015       P004         1  Cash on Delivery   
3    O0004  2026-03-04        C024       P015         3       Net Banking   
4    O0005  2026-03-29        C025       P009         5       Credit Card   

  Order_Status             Product_Name        Category  Unit_Price     Brand  \
0    Delivered              Cricket Bat          Sports        2499    BatPro   
1    Delivered           Wireless Mouse     Electronics         899  TechGear   
2    Delivered              Smart Watch     Electronics        3299   FitTech   
3    Delivered  Machine Learning Basics           Books         999   AIPress   
4    Delivered             Coffee Maker  Home & Kitchen        3499  HomeBrew   

   Customer_Name        Ci

In [3]:
# concat demonstration
sample1 = merged[['Order_ID', 'Customer_Name', 'Product_Name']].head(2)
sample2 = merged[['Order_ID', 'Customer_Name', 'Product_Name']].tail(2)
concat_df = pd.concat([sample1, sample2], ignore_index=True)
print("Concatenated samples:")
print(concat_df)

Concatenated samples:
  Order_ID  Customer_Name    Product_Name
0    O0001  Harsh Vardhan     Cricket Bat
1    O0002   Ishita Gupta  Wireless Mouse
2    O0119    Karan Joshi     Cricket Bat
3    O0120      Rahul Das    Dumbbell Set


In [4]:
# DateTime operations
merged['Order_Date'] = pd.to_datetime(merged['Order_Date'])
merged['Month'] = merged['Order_Date'].dt.month
merged['Day'] = merged['Order_Date'].dt.day
merged['DayOfWeek'] = merged['Order_Date'].dt.day_name()
print(merged[['Order_ID', 'Order_Date', 'Month', 'Day', 'DayOfWeek']].head())

  Order_ID Order_Date  Month  Day  DayOfWeek
0    O0001 2026-02-19      2   19   Thursday
1    O0002 2026-01-25      1   25     Sunday
2    O0003 2026-02-26      2   26   Thursday
3    O0004 2026-03-04      3    4  Wednesday
4    O0005 2026-03-29      3   29     Sunday


In [5]:
# apply() transformations
merged['Total_Value'] = merged.apply(lambda r: r['Quantity'] * r['Unit_Price'], axis=1)
merged['Is_Delivered'] = merged['Order_Status'].apply(lambda x: 1 if x == 'Delivered' else 0)

def get_segment(membership):
    if membership == 'Premium': return 'High'
    elif membership == 'Regular': return 'Medium'
    return 'Low'

merged['Customer_Segment'] = merged['Membership_Type'].apply(get_segment)

def price_cat(price):
    if price < 1000: return 'Budget'
    elif price < 2500: return 'Mid-Range'
    return 'Premium'

merged['Price_Category'] = merged['Unit_Price'].apply(price_cat)
print(merged[['Order_ID', 'Product_Name', 'Total_Value', 'Customer_Segment', 'Price_Category']].head())

  Order_ID             Product_Name  Total_Value Customer_Segment  \
0    O0001              Cricket Bat         4998             High   
1    O0002           Wireless Mouse         1798             High   
2    O0003              Smart Watch         3299           Medium   
3    O0004  Machine Learning Basics         2997           Medium   
4    O0005             Coffee Maker        17495              Low   

  Price_Category  
0      Mid-Range  
1         Budget  
2        Premium  
3         Budget  
4        Premium  


In [6]:
# Final processed dataset
final_df = merged[['Order_ID', 'Order_Date', 'Month', 'Day', 'DayOfWeek',
                   'Customer_ID', 'Customer_Name', 'City', 'Region', 'Membership_Type', 'Customer_Segment',
                   'Product_ID', 'Product_Name', 'Category', 'Unit_Price', 'Price_Category',
                   'Brand', 'Quantity', 'Total_Value', 'Payment_Method', 'Order_Status', 'Is_Delivered']]

print("Final shape:", final_df.shape)
print(final_df.head())

Final shape: (120, 22)
  Order_ID Order_Date  Month  Day  DayOfWeek Customer_ID  Customer_Name  \
0    O0001 2026-02-19      2   19   Thursday        C027  Harsh Vardhan   
1    O0002 2026-01-25      1   25     Sunday        C006   Ishita Gupta   
2    O0003 2026-02-26      2   26   Thursday        C015    Karan Joshi   
3    O0004 2026-03-04      3    4  Wednesday        C024    Maryam Khan   
4    O0005 2026-03-29      3   29     Sunday        C025   Reyansh Jain   

         City Region Membership_Type  ...             Product_Name  \
0       Noida  North         Premium  ...              Cricket Bat   
1   Bengaluru  South         Premium  ...           Wireless Mouse   
2  Chandigarh  North         Regular  ...              Smart Watch   
3   Hyderabad  South         Regular  ...  Machine Learning Basics   
4     Kolkata   East             New  ...             Coffee Maker   

         Category Unit_Price Price_Category     Brand Quantity Total_Value  \
0          Sports       249

In [7]:
# Summary statistics
print("Total Orders:", len(final_df))
print("Total Customers:", final_df['Customer_ID'].nunique())
print("Total Products:", final_df['Product_ID'].nunique())
print("Total Revenue: Rs.{:,.2f}".format(final_df['Total_Value'].sum()))
print("Avg Order Value: Rs.{:,.2f}".format(final_df['Total_Value'].mean()))
print("Delivered:", final_df['Is_Delivered'].sum())
print("Cancelled:", len(final_df) - final_df['Is_Delivered'].sum())

print("Category Revenue:")
print(final_df.groupby('Category')['Total_Value'].sum().sort_values(ascending=False))

print("Order Status:")
print(final_df['Order_Status'].value_counts())

Total Orders: 120
Total Customers: 30
Total Products: 20
Total Revenue: Rs.741,507.00
Avg Order Value: Rs.6,179.23
Delivered: 79
Cancelled: 41
Category Revenue:
Category
Home & Kitchen    243913
Electronics       177821
Clothing          168033
Sports            116736
Books              35004
Name: Total_Value, dtype: int64
Order Status:
Order_Status
Delivered    79
Cancelled    25
Shipped      16
Name: count, dtype: int64


In [8]:
# Export
final_df.to_csv('Processed_Ecommerce_Dataset.csv', index=False)
print("Saved as 'Processed_Ecommerce_Dataset.csv'")

Saved as 'Processed_Ecommerce_Dataset.csv'
